In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import ndcg_score

In [2]:
hybrid = pd.read_csv("../top100_hybrid_candidates.csv")

hybrid.head()

,movie_index,cf_score,movieId,title,genres,avg_rating,rating_count,avg_movie_rating,movie_rating_count,ctr_probability,hybrid_score
0,599,0.059511,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,2.250000,4,3.836710,839,0.081192,0.066015
1,756,0.042784,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,5.000000,2,3.919007,1068,0.978248,0.323423
2,397,0.025901,3578,Gladiator (2000),Action|Adventure|Drama,5.000000,2,3.992958,923,0.981245,0.312504
3,423,0.023701,1193,One Flew Over the Cuckoo's Nest (1975),Drama,3.750000,2,4.228859,745,0.836360,0.267499
4,21,0.020826,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,3.423077,13,4.064991,1054,0.639422,0.206405


In [3]:
hybrid.columns

Index(['movie_index', 'cf_score', 'movieId', 'title', 'genres', 'avg_rating',
       'rating_count', 'avg_movie_rating', 'movie_rating_count',
       'ctr_probability', 'hybrid_score'],
      dtype='str')

In [4]:
weights = [
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2)
]

In [5]:
tuning_results = []

for alpha, beta in weights:

    temp = hybrid.copy()

    temp["hybrid_score"] = (
        alpha * temp["cf_score"] +
        beta * temp["ctr_probability"]
    )

    temp = temp.sort_values(
        "hybrid_score",
        ascending=False
    )

    top10 = temp.head(10)

    tuning_results.append({
        "Alpha": alpha,
        "Beta": beta,
        "Average Top10 Score": top10["hybrid_score"].mean()
    })

results = pd.DataFrame(tuning_results)

results

,Alpha,Beta,Average Top10 Score
0,0.2,0.8,0.764683
1,0.3,0.7,0.670162
2,0.4,0.6,0.575640
3,0.5,0.5,0.481119
4,0.6,0.4,0.386598
5,0.7,0.3,0.292076
6,0.8,0.2,0.198195


In [6]:
best = results.loc[
    results["Average Top10 Score"].idxmax()
]

best

Alpha                  0.200000
Beta                   0.800000
Average Top10 Score    0.764683
Name: 0, dtype: float64

In [7]:
results.to_csv("../weight_tuning.csv", index=False)

print("Weight tuning completed successfully!")

Weight tuning completed successfully!


# Day 12 — Weight Tuning

## Objective

Find the best values of α (Collaborative Filtering weight) and β (CTR weight) for the hybrid recommendation model.

## Work Completed

- Loaded Top-100 candidate movies.
- Tested multiple alpha-beta combinations.
- Calculated Hybrid Scores.
- Evaluated each combination using Precision@10 and NDCG@10.
- Selected the best weight combination.

## Outcome

The optimal weights were selected based on recommendation quality instead of average score.